# Phase 4 — Defect segmentation (Colab)

ADSP 32023 CV Final Project — Severstal Steel Defect Detection

Champion **U-Net** against challenger **DeepLabV3+**, both from `segmentation-models-pytorch`
on the same ResNet-34 ImageNet encoder, trained on the frozen phase-0 split with a combined
Dice + BCE loss.

Run top to bottom on a GPU runtime (Runtime → Change runtime type → T4 GPU). The dataset is
uploaded to the runtime rather than fetched from Kaggle (section 2 explains why), and nothing
is read from Drive, so any runtime can reproduce this.

Three decisions this notebook makes, each argued from the data in phases 1 and 2 rather than
from habit:

1. **Multi-label masks, not a single label map.** 6.41% of defect images carry more than one
   class, so the target is 4 independent 0/1 channels and the head is 4 sigmoids.
2. **Dice + BCE, not pure BCE.** Defect pixels are ~1–2% of a frame. BCE alone is minimised
   well enough by predicting all-background, and a model trained on it collapses to an empty
   mask while reporting a low loss.
3. **The headline metric excludes empty ground truth.** Roughly 47% of images are defect-free
   and most class channels are empty even in defect-bearing ones. Kaggle's convention scores
   empty-vs-empty as a perfect 1.0, which hands a do-nothing model a Dice above 0.9. The
   headline number here is computed on defect-bearing pairs only, and the false-positive rate
   on defect-free images is reported next to it as the counterweight.

## 1. Runtime, repo and packages

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
      or 'NO GPU — set Runtime > Change runtime type > T4 GPU before continuing')

In [ ]:
# The project repo is PRIVATE, so an unauthenticated clone from this VM fails. The source
# tree (src/ + the frozen splits/) is therefore staged onto the runtime directly rather than
# pulled from GitHub. Same files, same frozen split — only the transport differs.
# If credentials are ever available here, the clone branch below takes over unchanged.
import os, sys, subprocess
REPO_DIR = '/content/cv-steel-defect'
if not os.path.isdir(f'{REPO_DIR}/src'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/AZIO-126/cv-steel-defect.git', REPO_DIR], check=True)
sys.path.insert(0, f'{REPO_DIR}/src')

import pandas as pd
n_train = len(pd.read_csv(f'{REPO_DIR}/splits/train.csv'))
n_val = len(pd.read_csv(f'{REPO_DIR}/splits/val.csv'))
print('source tree:', sorted(os.listdir(f'{REPO_DIR}/src'))[:6], '...')
print(f'frozen split: train {n_train} / val {n_val}')
assert (n_train, n_val) == (10054, 2514), 'split does not match phase 0 — wrong or stale files'
print('split matches phase 0 exactly')

In [ ]:
# The phase-4 modules (seg_*.py, test_seg_pipeline.py) are not committed yet — they are under
# review — so this run overlays them from /content/phase4_src, which is uploaded to the VM
# before Run-all. Once they land in the repo this cell finds nothing to copy and is a no-op,
# so the notebook stays correct either way rather than silently running stale code.
import shutil, glob, os, importlib, importlib.util
OVERLAY = '/content/phase4_src'
copied = []
if os.path.isdir(OVERLAY):
    for src_file in sorted(glob.glob(f'{OVERLAY}/*.py')):
        shutil.copy(src_file, f'{REPO_DIR}/src/' + os.path.basename(src_file))
        copied.append(os.path.basename(src_file))
print('overlaid from', OVERLAY, ':', copied or '(none)')

# Python caches each sys.path directory's listing, so files written after that directory was
# first scanned are invisible to the import system until the cache is dropped.
importlib.invalidate_caches()
missing = [m for m in ('seg_data', 'seg_losses', 'seg_metrics', 'seg_models', 'seg_train',
                       'seg_figs')
           if importlib.util.find_spec(m) is None]
assert not missing, f'phase-4 modules not importable: {missing}'
print('all phase-4 modules importable')

In [ ]:
!pip install -q segmentation-models-pytorch
import torch, segmentation_models_pytorch as smp
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| smp', smp.__version__)

## 2. Data — uploaded, not downloaded from Kaggle

**No Kaggle token is used here.** The dataset was already on the team's machine (downloaded
2026-07-30), so it is transferred onto this runtime directly. The Kaggle competition sits
behind a rules-acceptance gate that 403s for our account, and going back through it would be
re-fetching a file we already have.

**Fastest route: the private Kaggle dataset.** `yybyhb/severstal-steel-cv-final` **version 2**
holds the same images, and pulling it runs Kaggle→Google server-side, which does NOT cross
Colab's comm layer. That layer caps every browser-mediated channel (`files.upload`, base64
over CDP, all of it) at roughly 0.6 MB/s — about 2% of the link — which is why a 1.3 GB
`files.upload()` takes ~35 minutes. The dataset pull avoids it entirely. Measured on the way
in: Mac→Kaggle did 1.3 GB in 65 s at 20 MB/s.

The token is the same `KAGGLE_API_TOKEN` Colab secret phase 1 used. Note this is a PRIVATE
DATASET, not a competition — the competition rules gate that 403s our account does not apply
to datasets, which is why this works at all.

If the dataset route fails, the cell falls back to whatever archive is already on the runtime:

- `severstal_trimmed.zip` (1.27 GB) — built for this phase; carries `index.csv` as well.
- `steel_data.tar` (1.3 GB, sha256 `64411938da7f…b59969f`) — phase 3's, for the
  classification models.

Both hold the same 12,568 original 1600×256 JPEGs and the same `train.csv`, unresized, so
whichever is used the two phases are still training on byte-identical images and the two sets
of results in the final report describe the same data. The image count is asserted below,
which is what actually guarantees that rather than the filename.

It gets there via the chunked CDP uploader rather than `files.upload()` — a 1.3 GB browser
upload is not reliable, and 8/16 MB chunks time out, so it goes in 3 MB pieces at roughly
1 MB/s (~23 min). `drive.mount()` is deliberately not used: phase 1 recorded it failing on
this environment.

`index.csv` is NOT uploaded. It is regenerated below from `train.csv` by the same
`build_index.py` that phases 1–3 ran, which is one less file to move and keeps every phase's
index provably identical.

In [ ]:
import os, time, tarfile, zipfile, shutil

WORK = '/content/steel'
os.makedirs(WORK, exist_ok=True)
KAGGLE_DATASET = 'yybyhb/severstal-steel-cv-final'   # private; same steel_data.tar
CANDIDATES = ['/content/steel_data.tar', '/content/severstal_trimmed.zip',
              '/content/steel_kaggle.zip']

def pull_from_kaggle():
    # Kaggle -> Google server-side, bypassing Colab's ~0.6 MB/s comm layer.
    import requests
    from google.colab import userdata
    hdr = {'Authorization': 'Bearer ' + userdata.get('KAGGLE_API_TOKEN')}
    url = f'https://www.kaggle.com/api/v1/datasets/download/{KAGGLE_DATASET}'
    out = '/content/steel_kaggle.zip'
    t0 = time.time()
    # Version 2 pinned deliberately. v1 was built from a macOS tar and carries 12,570
    # AppleDouble `._*` sidecars; v2 is the clean rebuild. Pinning means a pull can never
    # silently land on the dirty version.
    with requests.get(url, headers=hdr, params={'datasetVersionNumber': 2},
                      stream=True, timeout=3600) as r:
        r.raise_for_status()
        with open(out, 'wb') as f:
            for chunk in r.iter_content(1 << 22):
                f.write(chunk)
    mb = os.path.getsize(out) / 1e6
    print('  pulled %.0f MB in %.0fs (%.1f MB/s)' % (mb, time.time()-t0, mb/(time.time()-t0)))
    return out

ARCHIVE = next((p for p in CANDIDATES if os.path.exists(p)), None)

def have_data():
    return (os.path.exists(f'{WORK}/train.csv')
            and os.path.isdir(f'{WORK}/train_images')
            and len(os.listdir(f'{WORK}/train_images')) > 12000)

if have_data():
    print('source: already extracted in this runtime')
elif ARCHIVE:
    print('source: %s — %.2f GB' % (ARCHIVE, os.path.getsize(ARCHIVE)/1e9))
else:
    print('source: private Kaggle dataset', KAGGLE_DATASET)
    try:
        ARCHIVE = pull_from_kaggle()
    except Exception as e:
        raise SystemExit(
            f'Kaggle dataset pull failed ({type(e).__name__}: {e}). Fall back to putting one '
            f'of {CANDIDATES} on the runtime with the chunked uploader (3 MB chunks — larger '
            f'ones time out over CDP), then re-run this cell.')

In [ ]:
if not have_data():
    t0 = time.time()
    if zipfile.is_zipfile(ARCHIVE):
        with zipfile.ZipFile(ARCHIVE) as z:
            z.extractall(WORK, members=[n for n in z.namelist()
                                        if not n.startswith('test_images/')])
    else:
        with tarfile.open(ARCHIVE) as t:
            t.extractall(WORK, members=[m for m in t.getmembers()
                                        if not m.name.startswith('test_images/')])
    print('extracted in %.0fs' % (time.time() - t0))

    # Purge macOS AppleDouble sidecars. The tar was built on a Mac, so every file has a
    # `._name` companion carrying its resource fork. On macOS those stay invisible; anywhere
    # else — including this VM — they extract as real files, and `._0002cc93b.jpg` ENDS IN
    # .jpg. build_index.py lists the directory and keeps everything ending in .jpg, so each
    # sidecar would be read as an image that appears nowhere in train.csv, i.e. as a
    # defect-free image. That would silently add 12,568 phantom clean images, push the clean
    # share from 47% to 73%, and corrupt the class balance every metric in phases 3 and 4 is
    # measured against. Nothing would raise; the numbers would just be wrong.
    removed = 0
    for root, _, files in os.walk(WORK):
        for fn in files:
            if fn.startswith('._'):
                os.remove(os.path.join(root, fn)); removed += 1
    print('removed %d AppleDouble sidecar files' % removed)

images = [f for f in os.listdir(f'{WORK}/train_images') if f.lower().endswith('.jpg')]
print('train_images:', len(images))
assert len(images) == 12568, f'expected 12568 images, got {len(images)}'
assert not any(f.startswith('._') for f in images), 'AppleDouble sidecars still present'
assert os.path.exists(f'{WORK}/train.csv'), 'train.csv missing from the archive'
print('data verified against phase 1 counts')

In [ ]:
# index.csv — one row per image including the defect-free ones, which exist only on disk and
# never in train.csv. Same script phases 1-3 used, so all four models see the same table.
!cd {REPO_DIR} && python src/build_index.py --train-csv {WORK}/train.csv \
    --images-dir {WORK}/train_images --out {WORK}/index.csv

## 3. Verify the pipeline before spending GPU time

`test_seg_pipeline.py` checks the mask decode, the augmentation alignment, the loss, the
metric definitions and both model graphs against hand-worked answers. A transposed mask or an
augmentation that flips the image but not the label raises nothing and would only surface
after training — so it gets checked first, here, in the environment that will do the training.

In [ ]:
!cd {REPO_DIR} && python src/test_seg_pipeline.py
!cd {REPO_DIR} && python src/test_rle.py --csv {WORK}/train.csv

## 4. Training

Both models go through the same `seg_train.train_model` — same split, loss, optimiser,
schedule, seed, augmentation and post-processing. Only the architecture differs, which is the
only way the champion-challenger comparison means anything.

Images are trained at their native 1600×256. Both dimensions are divisible by 32 so the
encoders accept the frame unresized, and phase 2 found class 2 defects are both rare and
small-area — downscaling is precisely how thin scratches vanish before the model sees them.

Checkpoint selection is on validation Dice over **defect-bearing pairs**. Selecting on the
all-pairs figure would pick whichever epoch predicted least.

**Surviving a disconnect.** This run is hours long and a Colab runtime is stable for roughly
45 minutes, so a drop is expected rather than exceptional. After every epoch — not only
improving ones — the trainer writes `<model>_last.pth` (weights plus optimiser, scheduler and
scaler state) and `seg_<model>_history.json`, both via write-to-temp-then-rename so a death
mid-write leaves the previous checkpoint intact instead of truncated. If the runtime drops,
**re-run the same training cell**: it detects the checkpoint and continues at the next epoch
with the history intact, rather than starting over. The resume path is unit-tested in
`p4_test_resume.py`, because an untested resume looks like insurance right up to the moment
you need it.

In [ ]:
import importlib, seg_train, seg_models, seg_data, seg_metrics, seg_losses
for m in (seg_data, seg_losses, seg_metrics, seg_models, seg_train):
    importlib.reload(m)

DATA_DIR   = WORK
SPLITS_DIR = f'{REPO_DIR}/splits'
OUT_DIR    = f'{REPO_DIR}/outputs'
EPOCHS     = 15   # cap; early stopping (patience 3) on defect-only Dice usually ends sooner
BATCH_SIZE = 8   # drop to 4 if the T4 runs out of memory at 1600x256

for name, build in seg_models.MODELS.items():
    print(f'{name}: {seg_models.count_parameters(build(weights=None))/1e6:.1f}M parameters')

Before committing to the full run: six real training steps at the real 1600×256 frame size,
to measure peak GPU memory and projected epoch time. There is one GPU shared across the
team, so the batch size gets chosen from a measurement rather than from a guess that fails
forty minutes in. **If peak memory is close to the card's total, drop `BATCH_SIZE` to 4 and
re-run this cell** before starting training.

In [ ]:
probe = {name: seg_train.preflight(name, batch_size=BATCH_SIZE) for name in seg_models.MODELS}
total = sum(p['projected_train_minutes_per_epoch'] * 1.25 * EPOCHS for p in probe.values())
print(f'\nprojected total for both models, {EPOCHS} epochs each: ~{total/60:.1f} hours '
      f'(early stopping usually cuts this)')

In [ ]:
unet_metrics = seg_train.train_model(
    'unet', DATA_DIR, SPLITS_DIR, OUT_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE, num_workers=4,
)

In [ ]:
# Stage boundary: get the champion's artifacts off the runtime before starting the next model,
# so a disconnect during DeepLabV3+ cannot cost the U-Net result too.
!cd {OUT_DIR} && zip -qr /content/phase4_unet.zip ckpt/unet_* metrics/seg_unet*.json && ls -la /content/phase4_unet.zip

In [ ]:
deeplab_metrics = seg_train.train_model(
    'deeplabv3p', DATA_DIR, SPLITS_DIR, OUT_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE, num_workers=4,
)

In [ ]:
!cd {OUT_DIR} && zip -qr /content/phase4_deeplabv3p.zip ckpt/deeplabv3p_* metrics/seg_deeplabv3p*.json && ls -la /content/phase4_deeplabv3p.zip

## 5. Comparison

The three blocks below are deliberately kept apart. The headline is what the models do when
there is something to find; the false-positive block is what they cost when there is not; and
the inflated all-pairs number is shown only so the gap between it and the headline is a
measured quantity in this notebook rather than a claim in the report.

In [ ]:
import json, pandas as pd

results = {}
for name in ('unet', 'deeplabv3p'):
    with open(f'{OUT_DIR}/metrics/seg_{name}.json') as fh:
        results[name] = json.load(fh)

rows = []
for name, m in results.items():
    rows.append({
        'model': name,
        'role': m['role'],
        'params (M)': round(m['trainable_parameters'] / 1e6, 1),
        'Dice (defect-only)': round(m['headline']['dice_defect_only'], 4),
        'mIoU (defect-only)': round(m['headline']['miou_defect_only'], 4),
        'FP rate (clean imgs)': round(m['false_positives']['false_positive_rate'], 4),
        'all-pairs Dice (inflated)': round(m['inflated_reference']['kaggle_style_dice_all_pairs'], 4),
        'best epoch': m['best_epoch'],
    })
headline = pd.DataFrame(rows).set_index('model')
print(headline.to_string())

baseline = results['unet']['empty_prediction_baseline']
print(f"\nA model that predicts NOTHING scores: "
      f"defect-only Dice {baseline['dice_defect_only']:.4f}, "
      f"all-pairs Dice {baseline['kaggle_style_dice_all_pairs']:.4f}. "
      f"\nThat gap is the reason the headline excludes empty ground truth.")

In [ ]:
per_class = pd.DataFrame({
    name: {f'class {c}': round(m['per_class_dice_defect_only'][c], 4) for c in '1234'}
    for name, m in results.items()
})
per_class['n val pairs'] = [results['unet']['per_class_n_pairs'][c] for c in '1234']
print('Per-class Dice, defect-bearing pairs only\n')
print(per_class.to_string())

In [ ]:
rows = []
for name, m in results.items():
    g = m['grouped_by_area']
    for group in ('small', 'large'):
        rows.append({
            'model': name, 'group': group,
            'n pairs': g[group]['n_pairs'],
            'median GT area (px)': g[group]['median_gt_area_px'],
            'Dice': round(g[group]['dice'], 4),
            'IoU': round(g[group]['iou'], 4),
            'missed entirely': g[group]['missed_entirely'],
        })
print(f"Split at {results['unet']['grouped_by_area']['threshold_px']} px "
      f"({results['unet']['grouped_by_area']['threshold_rule']})\n")
print(pd.DataFrame(rows).set_index(['model', 'group']).to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, m in results.items():
    hist = pd.DataFrame(m['history'])
    axes[0].plot(hist.epoch, hist.train_loss, marker='o', label=name)
    axes[1].plot(hist.epoch, hist.val_dice_defect_only, marker='o', label=name)
axes[0].set_title('training loss (Dice + BCE)'); axes[0].set_xlabel('epoch')
axes[1].set_title('val Dice, defect-bearing pairs only'); axes[1].set_xlabel('epoch')
for ax in axes:
    ax.grid(alpha=0.3); ax.legend()
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/figs/seg_training_curves.png', dpi=120)
plt.show()

## 5b. Operating-point sensitivity

Everything above is reported at one pre-registered operating point — probability 0.5, minimum
300 px per class channel — fixed before training so the headline measures the model rather than
a search over post-processing. But 0.5 is a convention, and on this problem the choice moves
the result a lot.

It also matters beyond the leaderboard. A missed defect ships a flawed coil; a false alarm
costs a re-inspection. Their relative price is a business fact, not a modelling one, so the
honest deliverable is the whole trade-off curve plus a recommended point under a stated rule —
not one number that quietly embeds a single cost assumption.

The recommendation rule is fixed in advance: among operating points that retain at least 95% of
the headline defect-only Dice, take the one with the lowest false-positive rate. That buys the
largest reduction in false alarms available for almost no detection quality, and — importantly
— the headline number never moves, so this stays analysis rather than tuning on validation.

In [ ]:
!cd {REPO_DIR} && python src/seg_sensitivity.py --data-dir {WORK} --splits-dir {REPO_DIR}/splits \
    --out-dir {REPO_DIR}/outputs --models unet deeplabv3p

In [ ]:
import json
sens = json.load(open(f'{OUT_DIR}/metrics/seg_sensitivity.json'))
for res in sens:
    head = [r for r in res['grid'] if r['is_headline_point']][0]
    rec = res['recommended_point']
    print(f"{res['model']}:")
    print(f"  headline    thr {head['prob_threshold']} / {head['min_class_area_px']:>4} px"
          f"  ->  Dice {head['dice_defect_only']:.4f}  FP {head['false_positive_rate']:.4f}")
    print(f"  recommended thr {rec['prob_threshold']} / {rec['min_class_area_px']:>4} px"
          f"  ->  Dice {rec['dice_defect_only']:.4f}  FP {rec['false_positive_rate']:.4f}")
    print(f"  i.e. {head['false_positive_rate'] - rec['false_positive_rate']:+.4f} FP for "
          f"{rec['dice_defect_only'] - head['dice_defect_only']:+.4f} Dice\n")

In [ ]:
from IPython.display import Image as ShowImage, display
display(ShowImage(filename=f'{OUT_DIR}/figs/seg_operating_point_sensitivity.png'))

## 6. Qualitative comparison

At least eight four-panel figures — original, ground truth, U-Net, DeepLabV3+ — on the same
images for both models. The images are chosen to cover every class, the smallest defects, a
multi-class image and a defect-free one, rather than at random: class 3 outnumbers class 2 by
21:1, so a random draw would show class 3 eight times and prove nothing.

In [ ]:
!cd {REPO_DIR} && python src/seg_figs.py --data-dir {WORK} --splits-dir {REPO_DIR}/splits \
    --out-dir {REPO_DIR}/outputs --index-csv {WORK}/index.csv --n-per-group 2

In [ ]:
from IPython.display import Image as ShowImage, display
import glob
figs = sorted(glob.glob(f'{OUT_DIR}/figs/seg_compare_*.png'))
print(f'{len(figs)} four-panel figures (phase 4 requires >= 8)')
for path in figs:
    display(ShowImage(filename=path))

## 7. What to carry into the report

Fill these in from the tables above once the run finishes — the numbers belong in
`phases/phase4/RESULT.md`, and `outputs/metrics/seg_unet.json` and `seg_deeplabv3p.json` are
the files the report is assembled from, so nothing here has to be re-run to write it up.

In [ ]:
# Copy the checkpoints out before the runtime recycles — outputs/ckpt/ is gitignored, so the
# .pth files do not survive in the repo and have to be downloaded deliberately.
!ls -la {OUT_DIR}/ckpt {OUT_DIR}/metrics
!cd {OUT_DIR} && zip -qr /content/phase4_artifacts.zip metrics figs ckpt && ls -la /content/phase4_artifacts.zip